# CNN-LSTM All-Feature Forecast Metrics Notebook

This notebook is self-contained for the all-feature forecasting run. It defines the local preprocessing, dataset windows, model, training loop, metrics, train-validation loss plot, and actual-vs-predicted feature plots directly in notebook cells.

The run uses the configured output horizon array and writes one result folder per target feature and horizon.


## 1. Import Libraries

These imports combine the loading, plotting, preprocessing, PyTorch model, dataloader, scaling, and metric logic from the original files.

In [1]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

## 2. Reproducibility and Paths

Use the current TSA project data folder. The cell checks both `indoorAir2.csv` and `indoorAir.csv` under `/Users/nidhimandaviya/Desktop/TSA/data`.


In [2]:
PROJECT_DIR = r"/Users/nidhimandaviya/Desktop/TSA"
DATA_CANDIDATES = [
    os.path.join(PROJECT_DIR, "data", "indoorAir2.csv"),
    os.path.join(PROJECT_DIR, "data", "indoorAir.csv"),
]

CSV_PATH = next((path for path in DATA_CANDIDATES if os.path.exists(path)), None)
if CSV_PATH is None:
    raise FileNotFoundError(f"Could not find any data file from: {DATA_CANDIDATES}")

print(CSV_PATH)


/Users/nidhimandaviya/Desktop/TSA/data/indoorAir2.csv


## 3. Notebook-Local Model Configuration

All values used by the metric runner are defined in this notebook. No `src` or pipeline files are imported.


In [3]:
BASE_FEATURES = [
    "ens160_aqi",
    "ens160_tvoc",
    "bme688_gas_resistance",
    "bme688_pressure",
    "scd41_temperature",
    "scd41_humidity",
    "scd41_co2",
    "hour_sin",
    "hour_cos",
    "dayofweek_sin",
    "dayofweek_cos",
    "is_weekend",
]

PCA_FEATURES = [
    "ens160_aqi",
    "ens160_tvoc",
    "bme688_gas_resistance",
    "bme688_pressure",
    "scd41_temperature",
    "scd41_humidity",
]

TARGET = "scd41_co2"
STATION_COLUMN = "station_id"
SEGMENT_COLUMN = "_continuous_segment_id"

INPUT_SEQ_LENGTH = 480
OUTPUT_LENGTHS = [1, 3, 6, 8, 12, 16, 24, 48, 56]
OUTPUT_SEQ_LENGTH = OUTPUT_LENGTHS[0]
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 1e-5
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT_RATE = 0.2
WEIGHT_DECAY = 1e-4
RESAMPLE_TIME = "15min"
CLIP_OUTLIERS = True
OUTLIER_CLIP_FACTOR = 1.5
RESTORE_BEST_MODEL = True

print("Input length:", INPUT_SEQ_LENGTH)
print("Forecast horizons:", OUTPUT_LENGTHS)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)


Input length: 480
Forecast horizons: [1, 3, 6, 8, 12, 16, 24, 48, 56]
Batch size: 64
Learning rate: 1e-05


## 4. Load and Prepare the Raw Dataset

This is the notebook version of `load_prepare_data`. It reads the CSV, converts Unix seconds to timestamps, sorts by time, and makes timestamp the index.

In [4]:
df = pd.read_csv(CSV_PATH)

print("========== RAW DATASET ==========")
print("Raw dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
df = df.sort_values(by="timestamp")
df = df.set_index("timestamp")

print("\n========== TIMESTAMP PROCESSING ==========")
print("Dataset shape:", df.shape)
print("\nDate Range:")
print("Start:", df.index.min())
print("End:", df.index.max())

df.head()

========== RAW DATASET ==========
Raw dataset shape: (1924653, 21)

Columns:
['id', 'station_id', 'station_name', 'timestamp', 'scd41_co2', 'scd41_temperature', 'scd41_humidity', 'ens160_eco2', 'ens160_tvoc', 'ens160_aqi', 'svm41_temperature', 'svm41_humidity', 'svm41_nox_index', 'svm41_voc_index', 'bme688_temperature', 'bme688_humidity', 'bme688_pressure', 'bme688_gas_resistance', 'sfa30_temperature', 'sfa30_humidity', 'sfa30_hco']

========== TIMESTAMP PROCESSING ==========
Dataset shape: (1924653, 20)

Date Range:
Start: 2024-11-13 16:57:56
End: 2026-04-22 13:10:59


,id,station_id,station_name,scd41_co2,scd41_temperature,scd41_humidity,ens160_eco2,ens160_tvoc,ens160_aqi,svm41_temperature,svm41_humidity,svm41_nox_index,svm41_voc_index,bme688_temperature,bme688_humidity,bme688_pressure,bme688_gas_resistance,sfa30_temperature,sfa30_humidity,sfa30_hco
timestamp,,,,,,,,,,,,,,,,,,,,
2024-11-13 16:57:56,1,1,station_room_233,488.0,22.951857,36.897277,418,32,1,23.684999,34.060001,29.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-11-13 16:58:57,2,1,station_room_233,485.0,22.335011,38.185119,414,30,1,23.590000,34.209999,79.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-11-13 16:59:59,3,1,station_room_233,477.0,22.353704,38.400268,443,43,1,23.655000,34.180000,99.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-11-13 17:01:03,4,1,station_room_233,474.0,22.417792,38.269042,406,26,1,23.709999,34.130001,93.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-11-13 17:02:05,5,1,station_room_233,473.0,22.556649,37.971496,416,31,1,23.659999,34.240001,98.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Quick Data Checks

Before modeling, inspect stations, missing values, and numeric summaries. This is useful in a notebook even though the original script mostly prints shapes.

In [5]:
print("Stations available:", sorted(df[STATION_COLUMN].dropna().unique().tolist()))
print("Total missing values:", df.isna().sum().sum())

df.describe(include="all").T.head(20)

Stations available: [1, 2, 3, 4, 5, 6]
Total missing values: 8501922


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,1924653.0,NaN,NaN,NaN,962329.700152,555600.075605,1.0,481167.0,962330.0,1443493.0,1924656.0
station_id,1924653.0,NaN,NaN,NaN,2.847304,1.447816,1.0,1.0,3.0,4.0,6.0
station_name,1924653,6,station_room_233,586547,NaN,NaN,NaN,NaN,NaN,NaN,NaN
scd41_co2,1924653.0,NaN,NaN,NaN,533.239516,233.990075,365.0,429.0,457.0,523.0,32774.0
scd41_temperature,1924653.0,NaN,NaN,NaN,24.66657,1.750211,15.936903,23.443198,24.51934,25.664911,130.0
scd41_humidity,1924653.0,NaN,NaN,NaN,35.090718,8.362188,14.282226,28.643798,33.636474,40.940856,99.998474
ens160_eco2,1924653.0,NaN,NaN,NaN,735.495093,3417.101072,0.0,451.0,494.0,578.0,65522.0
ens160_tvoc,1924653.0,NaN,NaN,NaN,111.988025,354.955305,0.0,47.0,69.0,117.0,65488.0
ens160_aqi,1924653.0,NaN,NaN,NaN,1.624931,0.7108,0.0,1.0,2.0,2.0,154.0
svm41_temperature,1155267.0,NaN,NaN,NaN,24.488481,1.842034,-0.005,23.29,24.29,25.344999,41.439998


## 9. Add Time Features

This is the exact feature engineering from `add_time_features`. Hour and day of week are encoded cyclically with sine/cosine, and weekends are marked with a binary flag.

In [9]:
def add_time_features(input_df):
    output_df = input_df.copy()

    hour = output_df.index.hour
    dayofweek = output_df.index.dayofweek

    output_df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    output_df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    output_df["dayofweek_sin"] = np.sin(2 * np.pi * dayofweek / 7)
    output_df["dayofweek_cos"] = np.cos(2 * np.pi * dayofweek / 7)
    output_df["is_weekend"] = (dayofweek >= 5).astype(int)

    return output_df

df_with_time = add_time_features(df)
df_with_time[BASE_FEATURES + [STATION_COLUMN]].head()

,ens160_aqi,ens160_tvoc,bme688_gas_resistance,bme688_pressure,scd41_temperature,scd41_humidity,scd41_co2,hour_sin,hour_cos,dayofweek_sin,dayofweek_cos,is_weekend,station_id
timestamp,,,,,,,,,,,,,
2024-11-13 16:57:56,1,32,NaN,NaN,22.951857,36.897277,488.0,-0.866025,-0.500000,0.974928,-0.222521,0,1
2024-11-13 16:58:57,1,30,NaN,NaN,22.335011,38.185119,485.0,-0.866025,-0.500000,0.974928,-0.222521,0,1
2024-11-13 16:59:59,1,43,NaN,NaN,22.353704,38.400268,477.0,-0.866025,-0.500000,0.974928,-0.222521,0,1
2024-11-13 17:01:03,1,26,NaN,NaN,22.417792,38.269042,474.0,-0.965926,-0.258819,0.974928,-0.222521,0,1
2024-11-13 17:02:05,1,31,NaN,NaN,22.556649,37.971496,473.0,-0.965926,-0.258819,0.974928,-0.222521,0,1


## 10. Select Model Features and Resample by Station

This is the key `preprocess_data` logic. After time features are added, the model keeps only `BASE_FEATURES` and `station_id`, then resamples each station independently to 15-minute intervals using mean values.

In [10]:
print("========== PREPROCESSING ==========")
print("Raw dataset shape:", df.shape)
print("Stations available:", sorted(df[STATION_COLUMN].unique().tolist()))

model_df = df_with_time[BASE_FEATURES + [STATION_COLUMN]].copy()

print("\nSelected features:")
print(BASE_FEATURES)
print("\nAfter feature selection:", model_df.shape)

model_df = (
    model_df
    .groupby(STATION_COLUMN)
    .resample("15min")
    .mean()
    .drop(columns=STATION_COLUMN, errors="ignore")
    .reset_index(level=0)
)

model_df = model_df[BASE_FEATURES + [STATION_COLUMN]]

print("\nAfter resampling:", model_df.shape)
model_df.head()

========== PREPROCESSING ==========
Raw dataset shape: (1924653, 20)
Stations available: [1, 2, 3, 4, 5, 6]

Selected features:
['ens160_aqi', 'ens160_tvoc', 'bme688_gas_resistance', 'bme688_pressure', 'scd41_temperature', 'scd41_humidity', 'scd41_co2', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'is_weekend']

After feature selection: (1924653, 13)

After resampling: (186894, 13)


,ens160_aqi,ens160_tvoc,bme688_gas_resistance,bme688_pressure,scd41_temperature,scd41_humidity,scd41_co2,hour_sin,hour_cos,dayofweek_sin,dayofweek_cos,is_weekend,station_id
timestamp,,,,,,,,,,,,,
2024-11-13 16:45:00,1.0,35.000000,NaN,NaN,22.546857,37.827555,483.333333,-0.866025,-0.500000,0.974928,-0.222521,0.0,1
2024-11-13 17:00:00,1.0,26.857143,NaN,NaN,22.746242,37.604195,472.285714,-0.965926,-0.258819,0.974928,-0.222521,0.0,1
2024-11-13 17:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2024-11-13 17:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2024-11-13 17:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


## 11. Shared Pipeline Train, Validation, and Test Split

This notebook now follows the shared CNN-LSTM preprocessing module. The split is chronological within each station through `train_val_test_spliting(...)`, so every station can contribute train, validation, and test rows. It no longer uses fixed train/validation/test station IDs.


In [11]:
def train_val_test_spliting(
    feature_df,
    station_column=STATION_COLUMN,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
):
    print("\n========== Train, Validation and Test ==========")
    feature_df = feature_df.copy()

    if "timestamp" not in feature_df.columns:
        feature_df["timestamp"] = feature_df.index

    feature_df = feature_df.reset_index(drop=True)
    feature_df["timestamp"] = pd.to_datetime(feature_df["timestamp"])
    feature_df = feature_df.sort_values("timestamp").reset_index(drop=True)

    n_rows = len(feature_df)
    train_end = int(n_rows * train_ratio)
    val_end = train_end + int(n_rows * val_ratio)

    train_df = feature_df.iloc[:train_end].copy().reset_index(drop=True)
    val_df = feature_df.iloc[train_end:val_end].copy().reset_index(drop=True)
    test_df = feature_df.iloc[val_end:].copy().reset_index(drop=True)

    total_rows = len(feature_df)
    print("Train shape:", train_df.shape)
    print("Validation shape:", val_df.shape)
    print("Test shape:", test_df.shape)
    print("Train percentage:", len(train_df) / total_rows * 100)
    print("Validation percentage:", len(val_df) / total_rows * 100)
    print("Test percentage:", len(test_df) / total_rows * 100)

    return train_df, val_df, test_df


train_df, val_df, test_df = train_val_test_spliting(
    model_df,
    station_column=STATION_COLUMN,
)

print("========== NOTEBOOK LOCAL SPLIT ==========")
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

train_parts = [train_df]
val_parts = [val_df]
test_parts = [test_df]



========== Train, Validation and Test ==========
Train shape: (130825, 14)
Validation shape: (28034, 14)
Test shape: (28035, 14)
Train percentage: 69.99957194987533
Validation percentage: 14.999946493734416
Test percentage: 15.000481556390252
========== NOTEBOOK LOCAL SPLIT ==========
Train shape: (130825, 14)
Validation shape: (28034, 14)
Test shape: (28035, 14)


### Reference Only: Old Split Idea

This reference cell is kept only for comparison with older notebook logic. The active workflow now uses the shared pipeline split above.


In [12]:
# Reference only: not used by the active CNN-LSTM pipeline.
#
# train_parts, val_parts, test_parts = [], [], []
# for station_id, station_df in model_df.groupby(STATION_COLUMN):
#     station_df = station_df.sort_index()
#     train_end = int(len(station_df) * 0.70)
#     val_end = int(len(station_df) * 0.85)
#
#     station_train_df = station_df.iloc[:train_end]
#     station_val_df = station_df.iloc[train_end:val_end]
#     station_test_df = station_df.iloc[val_end:]
#
#     train_parts.append(station_train_df)
#     val_parts.append(station_val_df)
#     test_parts.append(station_test_df)

## 12. Fill Missing Values

This follows `fill_missing_parts`. Rows with missing target values are removed first. Other input features are forward-filled and backward-filled inside each split part, and any remaining missing rows are dropped.

In [13]:
def fill_missing_parts(parts):
    cleaned_parts = []
    input_features = [col for col in BASE_FEATURES if col != TARGET]

    for part in parts:
        part = part.copy()

        print("\nMissing values before interpolation:")
        print(part.isna().sum().sum())

        part = part.dropna(subset=[TARGET])
        part[input_features] = part[input_features].ffill().bfill()
        part = part.dropna()

        print("Missing values after interpolation:")
        print(part.isna().sum().sum())

        if len(part) > 0:
            cleaned_parts.append(part)

    return cleaned_parts

train_parts = fill_missing_parts(train_parts)
val_parts = fill_missing_parts(val_parts)
test_parts = fill_missing_parts(test_parts)

[len(p) for p in train_parts], [len(p) for p in val_parts], [len(p) for p in test_parts]


Missing values before interpolation:
511950
Missing values after interpolation:
0

Missing values before interpolation:
84108
Missing values after interpolation:
0

Missing values before interpolation:
19908
Missing values after interpolation:
0


([88658], [21025], [26376])

## 13. Add One-Hot Station Features

The original model keeps `station_id` out of the numeric model array and instead creates one-hot station columns for every station found after preprocessing.

In [14]:
station_ids = sorted(model_df[STATION_COLUMN].unique().tolist())
station_features = [f"station_{sid}" for sid in station_ids]
model_features = BASE_FEATURES + station_features

def build_model_parts(parts, station_ids):
    station_features = [f"station_{sid}" for sid in station_ids]
    model_features = BASE_FEATURES + station_features
    model_parts = []

    for part in parts:
        part = part.copy()

        for sid in station_ids:
            part[f"station_{sid}"] = (part[STATION_COLUMN] == sid).astype(int)

        values = part[model_features].values

        if len(values) > 0:
            model_parts.append(values)

    return model_parts, model_features

train_arrays, model_features = build_model_parts(train_parts, station_ids)
val_arrays, _ = build_model_parts(val_parts, station_ids)
test_arrays, _ = build_model_parts(test_parts, station_ids)

print("Station IDs:", station_ids)
print("Model features:", model_features)
print("Number of model features:", len(model_features))

Station IDs: [1, 2, 3, 4, 5, 6]
Model features: ['ens160_aqi', 'ens160_tvoc', 'bme688_gas_resistance', 'bme688_pressure', 'scd41_temperature', 'scd41_humidity', 'scd41_co2', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'is_weekend', 'station_1', 'station_2', 'station_3', 'station_4', 'station_5', 'station_6']
Number of model features: 18


## 14. Scale Data with Training Data Only

This is an important leakage-prevention step. The `MinMaxScaler` is fit only on the training arrays, then applied to train, validation, and test arrays.

In [15]:
scaler = MinMaxScaler()
scaler.fit(np.vstack(train_arrays))

print("Scaler fitted ONLY on training data.")

train_scaled = [scaler.transform(p) for p in train_arrays if len(p) > 0]
val_scaled = [scaler.transform(p) for p in val_arrays if len(p) > 0]
test_scaled = [scaler.transform(p) for p in test_arrays if len(p) > 0]

train_scaled[0].shape, val_scaled[0].shape, test_scaled[0].shape

Scaler fitted ONLY on training data.


((88658, 18), (21025, 18), (26376, 18))

## 15. Create Multi-Step Forecasting Sequences

Each input sample uses `INPUT_SEQ_LENGTH=24` time steps. Each target contains the next `OUTPUT_SEQ_LENGTH=6` CO2 values. The target index is located inside `model_features`, so the target remains correct even after station one-hot features are added.

In [16]:
def create_sequences(data_parts, model_features, input_seq_length=24, output_seq_length=6):
    X, y = [], []
    target_index = model_features.index(TARGET)

    for data in data_parts:
        print("\nOriginal data shape before sequencing:", data.shape)

        required_length = input_seq_length + output_seq_length

        if len(data) <= required_length:
            print("\nSkipping sequence generation:")
            print(f"Available rows: {len(data)}")
            print(f"Required minimum rows: {required_length + 1}")
            continue

        for i in range(len(data) - input_seq_length - output_seq_length):
            X.append(data[i:i + input_seq_length])
            y.append(data[i + input_seq_length:i + input_seq_length + output_seq_length, target_index])

    X = np.array(X)
    y = np.array(y)

    print("Sequence input shape:", X.shape)
    print("Sequence target shape:", y.shape)

    if len(X) == 0:
        raise ValueError(
            "\nNo CNN-LSTM sequences were created.\n"
            "Possible reasons:\n"
            "- selected station has too few rows\n"
            "- too many missing values removed\n"
            "- input/output sequence lengths are too large\n"
            "- train/validation/test split is too small"
        )

    return X, y

X_train, y_train = create_sequences(train_scaled, model_features, INPUT_SEQ_LENGTH, OUTPUT_SEQ_LENGTH)
X_val, y_val = create_sequences(val_scaled, model_features, INPUT_SEQ_LENGTH, OUTPUT_SEQ_LENGTH)
X_test, y_test = create_sequences(test_scaled, model_features, INPUT_SEQ_LENGTH, OUTPUT_SEQ_LENGTH)


Original data shape before sequencing: (88658, 18)
Sequence input shape: (88177, 480, 18)
Sequence target shape: (88177, 1)

Original data shape before sequencing: (21025, 18)
Sequence input shape: (20544, 480, 18)
Sequence target shape: (20544, 1)

Original data shape before sequencing: (26376, 18)
Sequence input shape: (25895, 480, 18)
Sequence target shape: (25895, 1)


## 16. Build PyTorch DataLoaders

The training loader shuffles batches. Validation and test loaders keep order stable.

In [17]:
def create_loader(X, y, batch_size=BATCH_SIZE, shuffle=False):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32)

    loader = DataLoader(
        TensorDataset(X_tensor, y_tensor),
        batch_size=batch_size,
        shuffle=shuffle,
    )

    sample_X, sample_y = next(iter(loader))

    print("\n========== DATALOADER ==========")
    print("Batch input shape:", sample_X.shape)
    print("Batch target shape:", sample_y.shape)

    return loader

train_loader = create_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
val_loader = create_loader(X_val, y_val, BATCH_SIZE, shuffle=False)
test_loader = create_loader(X_test, y_test, BATCH_SIZE, shuffle=False)

input_size = X_train.shape[2]
input_size



========== DATALOADER ==========
Batch input shape: torch.Size([64, 480, 18])
Batch target shape: torch.Size([64, 1])

========== DATALOADER ==========
Batch input shape: torch.Size([64, 480, 18])
Batch target shape: torch.Size([64, 1])

========== DATALOADER ==========
Batch input shape: torch.Size([64, 480, 18])
Batch target shape: torch.Size([64, 1])


18

## 17. Define the CNN-LSTM Model

This is the model class from the source file. The input is shaped as `batch, time, features`. The model permutes it for 1D convolutions, applies two convolution layers with ReLU and dropout, permutes back for the LSTM, takes the final LSTM time step, and predicts all future CO2 steps with one linear layer.

In [18]:
class CNNLSTMModel(nn.Module):
    def __init__(self, input_size, output_seq_length=6, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()

        self.conv1 = nn.Conv1d(input_size, 128, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            128,
            hidden_size,
            num_layers,
            batch_first=True,
        )

        self.fc = nn.Linear(hidden_size, output_seq_length)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.dropout(self.relu(self.conv1(x)))
        x = self.dropout(self.relu(self.conv2(x)))
        x = x.permute(0, 2, 1)

        output, _ = self.lstm(x)
        output = output[:, -1, :]

        return self.fc(output)

model = CNNLSTMModel(input_size=input_size, output_seq_length=OUTPUT_SEQ_LENGTH)
model

CNNLSTMModel(
  (conv1): Conv1d(18, 128, kernel_size=(5,), stride=(1,), padding=(2,))
  (conv2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (lstm): LSTM(128, 64, num_layers=2, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

## 18. Sanity Check One Forward Pass

Before training, check that one batch produces `OUTPUT_SEQ_LENGTH` predictions per sample.

In [19]:
sample_X, sample_y = next(iter(train_loader))
with torch.no_grad():
    sample_predictions = model(sample_X)

print("Sample input shape:", sample_X.shape)
print("Sample target shape:", sample_y.shape)
print("Sample prediction shape:", sample_predictions.shape)

Sample input shape: torch.Size([64, 480, 18])
Sample target shape: torch.Size([64, 1])
Sample prediction shape: torch.Size([64, 1])


## 19. Training Helpers

These cells keep the original loss and optimizer choices: MSE loss and Adam with learning rate `0.001`.

In [20]:
def evaluate_loss(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            total_loss += loss.item()

    return total_loss / len(loader)


def train_model(model, train_loader, val_loader, epochs=10):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss = train_loss / len(train_loader)
        val_loss = evaluate_loss(model, val_loader, criterion)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(
            f"Epoch {epoch + 1}, "
            f"Train Loss: {train_loss:.6f}, "
            f"Val Loss: {val_loss:.6f}"
        )

    return model, train_losses, val_losses

## 20. Train the Model

Run this cell to train the CNN-LSTM. Increase `EPOCHS` in the configuration cell if you want a longer run.

In [ ]:
model, train_losses, val_losses = train_model(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)

## 21. Plot Training and Validation Loss

This is the notebook version of `plot_loss_curves`.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss")
plt.legend()
plt.show()

## 22. Evaluate on the Test Station

This follows `evaluate_model`. The metrics are computed after flattening all forecast steps together, exactly like the source code.

In [ ]:
def evaluate_model(model, test_loader):
    model.eval()
    predictions = []
    actuals = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            predictions.extend(outputs.numpy())
            actuals.extend(y_batch.numpy())

    predictions = np.array(predictions)
    actuals = np.array(actuals)

    mse = mean_squared_error(actuals.flatten(), predictions.flatten())
    mae = mean_absolute_error(actuals.flatten(), predictions.flatten())
    rmse = np.sqrt(mse)

    print("\n========== MODEL EVALUATION ==========")
    print("Overall MSE:", mse)
    print("Overall MAE:", mae)
    print("Overall RMSE:", rmse)

    return predictions, actuals, mse, mae, rmse

predictions, actuals, mse, mae, rmse = evaluate_model(model, test_loader)

## 23. Optional Per-Step Forecast Metrics

The per-step metric block was commented out in the source. It is useful in a notebook because this is a multi-step forecast, so each forecast horizon can behave differently.

In [ ]:
for step in range(actuals.shape[1]):
    step_mse = mean_squared_error(actuals[:, step], predictions[:, step])
    step_mae = mean_absolute_error(actuals[:, step], predictions[:, step])
    step_rmse = np.sqrt(step_mse)

    print(f"Forecast Step {step + 1}")
    print(f"MSE: {step_mse:.6f}")
    print(f"MAE: {step_mae:.6f}")
    print(f"RMSE: {step_rmse:.6f}\n")

## 24. Plot Actual vs Predicted CO2

This matches `plot_predictions`. Change `FORECAST_STEP_TO_PLOT` from 1 to 6 to inspect different forecast horizons.

In [ ]:
FORECAST_STEP_TO_PLOT = 1
step_index = FORECAST_STEP_TO_PLOT - 1

plt.figure(figsize=(10, 4))
plt.plot(actuals[:, step_index], label="Actual")
plt.plot(predictions[:, step_index], label="Predicted")
plt.legend()
plt.title(f"Actual vs Predicted CO2 for CNN-LSTM (Forecast Step {FORECAST_STEP_TO_PLOT})")
plt.xlabel("Test sequence")
plt.ylabel("Scaled CO2")
plt.show()

## 25. Collect Results

This mirrors the dictionary returned by `run_cnn_lstm_model`.

In [ ]:
cnn_results = {
    "model": model,
    "predictions": predictions,
    "actuals": actuals,
    "mse": mse,
    "mae": mae,
    "rmse": rmse,
}

cnn_results.keys()

## 26. Optional: Save the Trained Model

The original script returns the model but does not save it. This optional cell saves the trained PyTorch weights if you want to reuse them later.

In [ ]:
# Optional save. Uncomment when needed.
# MODEL_OUTPUT_PATH = os.path.join(PROJECT_DIR, "notebooks", "cnn_lstm_model.pt")
# torch.save(model.state_dict(), MODEL_OUTPUT_PATH)
# MODEL_OUTPUT_PATH

## 27. Compact Function Version

After running the notebook step by step, this helper gives you a compact function similar to the original `run_pipeline`, while still using the notebook-defined pieces above.

In [ ]:
def run_notebook_pipeline(raw_df, epochs=EPOCHS, input_seq_length=INPUT_SEQ_LENGTH, output_seq_length=OUTPUT_SEQ_LENGTH, batch_size=BATCH_SIZE):
    prepared = add_time_features(raw_df)
    prepared = prepared[BASE_FEATURES + [STATION_COLUMN]]
    prepared = (
        prepared
        .groupby(STATION_COLUMN)
        .resample(RESAMPLE_TIME)
        .mean()
        .drop(columns=STATION_COLUMN, errors="ignore")
        .reset_index(level=0)
    )
    prepared = prepared[BASE_FEATURES + [STATION_COLUMN]]

    station_ids = sorted(prepared[STATION_COLUMN].unique().tolist())
    train_df, val_df, test_df = train_val_test_spliting(prepared, station_column=STATION_COLUMN)

    train_parts = fill_missing_parts([train_df])
    val_parts = fill_missing_parts([val_df])
    test_parts = fill_missing_parts([test_df])

    train_arrays, model_features = build_model_parts(train_parts, station_ids)
    val_arrays, _ = build_model_parts(val_parts, station_ids)
    test_arrays, _ = build_model_parts(test_parts, station_ids)

    scaler = MinMaxScaler()
    scaler.fit(np.vstack(train_arrays))

    train_scaled = [scaler.transform(p) for p in train_arrays if len(p) > 0]
    val_scaled = [scaler.transform(p) for p in val_arrays if len(p) > 0]
    test_scaled = [scaler.transform(p) for p in test_arrays if len(p) > 0]

    X_train, y_train = create_sequences(train_scaled, model_features, input_seq_length, output_seq_length)
    X_val, y_val = create_sequences(val_scaled, model_features, input_seq_length, output_seq_length)
    X_test, y_test = create_sequences(test_scaled, model_features, input_seq_length, output_seq_length)

    train_loader = create_loader(X_train, y_train, batch_size, shuffle=True)
    val_loader = create_loader(X_val, y_val, batch_size, shuffle=False)
    test_loader = create_loader(X_test, y_test, batch_size, shuffle=False)

    model = CNNLSTMModel(input_size=X_train.shape[2], output_seq_length=output_seq_length)
    model, train_losses, val_losses = train_model(model, train_loader, val_loader, epochs=epochs)
    predictions, actuals, mse, mae, rmse = evaluate_model(model, test_loader)

    return {
        "model": model,
        "predictions": predictions,
        "actuals": actuals,
        "mse": mse,
        "mae": mae,
        "rmse": rmse,
        "train_losses": train_losses,
        "val_losses": val_losses,
    }


## 28. Feature Batch CNN-LSTM Pipeline

Clean notebook-local pipeline for selected features. Results are saved together in one folder, not one folder per feature or horizon.


### 28.1 Libraries, Paths, and Configuration


In [ ]:
# Notebook-only feature batch pipeline. No src or pipeline imports.
import copy
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

print("Libraries imported..!")


def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data" / "indoorAir2.csv").exists():
            return candidate
    return Path("/Users/nidhimandaviya/Desktop/TSA")


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

MPL_CONFIG_DIR = PROJECT_ROOT / ".matplotlib"
MPL_CONFIG_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CONFIG_DIR))

CSV_PATH = PROJECT_ROOT / "data" / "indoorAir2.csv"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "LSTM and CNN_LSTM Notebook" / "LSTM-CNN-NID" / "feature_batch_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_FEATURES = [
    "ens160_aqi",
    "ens160_tvoc",
    "bme688_gas_resistance",
    "bme688_pressure",
    "scd41_temperature",
    "scd41_humidity",
    "scd41_co2",
    "hour_sin",
    "hour_cos",
    "dayofweek_sin",
    "dayofweek_cos",
    "is_weekend",
]
SENSOR_FEATURES = [
    "ens160_aqi",
    "ens160_tvoc",
    "bme688_gas_resistance",
    "bme688_pressure",
    "scd41_temperature",
    "scd41_humidity",
    "scd41_co2",
]
TARGET_FEATURES = [
    "ens160_aqi",
    "ens160_tvoc",
    "bme688_gas_resistance",
    "bme688_pressure",
    "scd41_temperature",
    "scd41_humidity",
    "scd41_co2",
]
RUN_TARGET_FEATURES = ["scd41_co2"]

TARGET = "scd41_co2"
STATION_COLUMN = "station_id"
SEGMENT_COLUMN = "_continuous_segment_id"

INPUT_SEQ_LENGTH = 480
OUTPUT_LENGTHS = [1, 3, 6, 8, 12, 16, 24, 48, 56]
OUTPUT_SEQ_LENGTH = OUTPUT_LENGTHS[0]
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 1e-5
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT_RATE = 0.2
WEIGHT_DECAY = 1e-4
RESAMPLE_TIME = "15min"
MAX_FILL_STEPS = 2
DROP_SHORT_STATIONS = True
CLIP_OUTLIERS = True
OUTLIER_CLIP_FACTOR = 1.5
RESTORE_BEST_MODEL = True
USE_GAP_AWARE_SEGMENTS = False
USE_SCATTERING = False
USE_ATTENTION = False
DEVICE_PREFERENCE = "mps"
SHOW_PLOTS_IN_NOTEBOOK = True
PLOT_MAX_TIMESTAMPS = 1000
PLOT_ALL_TIMESTAMPS_FOR_CO2 = True
FUTURE_REFERENCE_STEPS = 3
FUTURE_REFERENCE_MAX_ROWS = 1000

CONV_CHANNELS = 96


### 28.2 Data Loading and Processing


In [ ]:
def load_prepare_data(csv_path):
    print("\n========== Data Loading ==========")
    df = pd.read_csv(csv_path)
    print(df.shape)
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df


def add_time_features(df):
    df = df.copy()
    timestamps = pd.to_datetime(df["timestamp"])
    hour = timestamps.dt.hour
    dayofweek = timestamps.dt.dayofweek
    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    df["dayofweek_sin"] = np.sin(2 * np.pi * dayofweek / 7)
    df["dayofweek_cos"] = np.cos(2 * np.pi * dayofweek / 7)
    df["is_weekend"] = (dayofweek >= 5).astype(int)
    return df


def preprocess_data(df):
    print("\n========== Feature Selecting ==========")
    df = df.copy()
    if "timestamp" not in df.columns:
        df["timestamp"] = df.index
    df = df.reset_index(drop=True)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    sensor_features = [column for column in SENSOR_FEATURES if column in df.columns]
    df = df[df[STATION_COLUMN] != 6].copy()
    df = df[sensor_features + ["timestamp", STATION_COLUMN]]
    print(df.shape)

    label = "15 Minutes" if str(RESAMPLE_TIME).lower() == "15min" else RESAMPLE_TIME
    print(f"\n========== Data Resampling with {label} ==========")
    df = (
        df.sort_values([STATION_COLUMN, "timestamp"])
        .set_index("timestamp")
        .groupby(STATION_COLUMN)[sensor_features]
        .resample(RESAMPLE_TIME)
        .mean()
        .reset_index()
    )
    print(df.shape)
    df = add_time_features(df)
    return df[BASE_FEATURES + ["timestamp", STATION_COLUMN]]


def add_continuous_segments(df):
    df = df.copy().sort_values([STATION_COLUMN, "timestamp"]).reset_index(drop=True)
    expected_delta = pd.to_timedelta(RESAMPLE_TIME)
    segment_parts = []
    for _, station_data in df.groupby(STATION_COLUMN, sort=True):
        station_data = station_data.copy()
        timestamp_diff = station_data["timestamp"].diff()
        station_data[SEGMENT_COLUMN] = (timestamp_diff > expected_delta).cumsum().astype(int)
        segment_parts.append(station_data)
    return pd.concat(segment_parts, ignore_index=True)


def fill_missing_dataframe(df):
    print("\n========== Data Gap Handling ==========")
    print("Expected timestamp interval:", RESAMPLE_TIME)
    print("Gap-aware sequence generation:", USE_GAP_AWARE_SEGMENTS)
    print("Sequences crossing detected timestamp gaps:", "disabled" if USE_GAP_AWARE_SEGMENTS else "allowed")

    print("\n========== Missing Values ==========")
    df = df.copy().sort_values([STATION_COLUMN, "timestamp"]).reset_index(drop=True)
    if USE_GAP_AWARE_SEGMENTS:
        df = add_continuous_segments(df)
    else:
        df[SEGMENT_COLUMN] = 0

    numeric_columns = [column for column in BASE_FEATURES if column in df.columns]
    for _, segment_index in df.groupby([STATION_COLUMN, SEGMENT_COLUMN]).groups.items():
        idx = list(segment_index)
        df.loc[idx, numeric_columns] = (
            df.loc[idx, numeric_columns]
            .interpolate(method="linear", limit=MAX_FILL_STEPS, limit_direction="both")
            .ffill(limit=MAX_FILL_STEPS)
            .bfill(limit=MAX_FILL_STEPS)
        )
    df[numeric_columns] = df[numeric_columns].ffill().bfill()
    print(df.shape)
    return df


def clip_outliers_dataframe(df):
    print("\n========== Clipping Outliers ==========")
    print("Clip factor:", OUTLIER_CLIP_FACTOR)
    df = df.copy()
    numeric_columns = [column for column in BASE_FEATURES if column in df.columns]
    for column in numeric_columns:
        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - OUTLIER_CLIP_FACTOR * iqr
        upper = q3 + OUTLIER_CLIP_FACTOR * iqr
        df[column] = df[column].clip(lower, upper)
    print(df.shape)
    return df


def drop_short_stations_for_windowing(feature_df):
    if not DROP_SHORT_STATIONS:
        return feature_df
    required_rows = INPUT_SEQ_LENGTH + max(OUTPUT_LENGTHS)
    station_counts = feature_df.groupby(STATION_COLUMN).size()
    keep_ids = station_counts[station_counts >= required_rows].index.tolist()
    return feature_df[feature_df[STATION_COLUMN].isin(keep_ids)].copy().reset_index(drop=True)


def train_val_test_spliting(feature_df, train_ratio=0.70, val_ratio=0.15):
    print("\n========== Train, Validation and Test ==========")
    feature_df = feature_df.copy().reset_index(drop=True)
    train_parts, val_parts, test_parts = [], [], []

    for _, station_data in feature_df.groupby(STATION_COLUMN, sort=True):
        station_data = station_data.sort_values("timestamp")
        n_rows = len(station_data)
        train_end = int(n_rows * train_ratio)
        val_end = train_end + int(n_rows * val_ratio)
        train_parts.append(station_data.iloc[:train_end])
        val_parts.append(station_data.iloc[train_end:val_end])
        test_parts.append(station_data.iloc[val_end:])

    train_df = pd.concat(train_parts).sort_values([STATION_COLUMN, "timestamp"]).reset_index(drop=True)
    val_df = pd.concat(val_parts).sort_values([STATION_COLUMN, "timestamp"]).reset_index(drop=True)
    test_df = pd.concat(test_parts).sort_values([STATION_COLUMN, "timestamp"]).reset_index(drop=True)
    total_rows = len(feature_df)

    print("Train shape:", train_df.shape)
    print("Val shape:", val_df.shape)
    print("Test shape:", test_df.shape)
    print("Train percentage:", len(train_df) / total_rows * 100)
    print("Val percentage:", len(val_df) / total_rows * 100)
    print("Test percentage:", len(test_df) / total_rows * 100)

    split_counts = pd.concat(
        [
            train_df.groupby(STATION_COLUMN).size().rename("train"),
            val_df.groupby(STATION_COLUMN).size().rename("val"),
            test_df.groupby(STATION_COLUMN).size().rename("test"),
        ],
        axis=1,
    ).fillna(0).astype(int)
    print("\nRows per station:")
    print(split_counts)
    return train_df, val_df, test_df


def build_future_target_reference(feature_df):
    print("\n========== Future Target Reference ==========")
    print(f"Creating ahead target columns for analysis only: step 1 to step {FUTURE_REFERENCE_STEPS}")
    print("These columns are not used as model input features.")
    reference_parts = []
    for _, station_data in feature_df.groupby(STATION_COLUMN, sort=True):
        station_data = station_data.sort_values("timestamp").copy()
        for step in range(1, FUTURE_REFERENCE_STEPS + 1):
            station_data[f"{TARGET}_ahead_{step}"] = station_data[TARGET].shift(-step)
        reference_parts.append(station_data[["timestamp", STATION_COLUMN, TARGET] + [f"{TARGET}_ahead_{step}" for step in range(1, FUTURE_REFERENCE_STEPS + 1)]])
    reference_df = pd.concat(reference_parts, ignore_index=True).dropna().head(FUTURE_REFERENCE_MAX_ROWS)
    print("Future target reference shape:", reference_df.shape)
    return reference_df


def add_station_features(df, station_ids):
    df = df.copy()
    for station_id in station_ids:
        df[f"station_{station_id}"] = (df[STATION_COLUMN] == station_id).astype(float)
    return df


def prepare_base_data(raw_df):
    feature_df = preprocess_data(raw_df)
    feature_df = drop_short_stations_for_windowing(feature_df)
    station_ids = sorted(feature_df[STATION_COLUMN].unique().tolist())
    feature_df = fill_missing_dataframe(feature_df)
    if CLIP_OUTLIERS:
        feature_df = clip_outliers_dataframe(feature_df)

    train_df, val_df, test_df = train_val_test_spliting(feature_df)
    future_target_reference = build_future_target_reference(feature_df)

    train_df = add_station_features(train_df, station_ids)
    val_df = add_station_features(val_df, station_ids)
    test_df = add_station_features(test_df, station_ids)
    station_features = [f"station_{station_id}" for station_id in station_ids]
    model_features = BASE_FEATURES + station_features

    print("\n========== Min Max Scaling ==========")
    scaler = MinMaxScaler()
    scaler.fit(train_df[model_features])
    train_df[model_features] = scaler.transform(train_df[model_features])
    val_df[model_features] = scaler.transform(val_df[model_features])
    test_df[model_features] = scaler.transform(test_df[model_features])
    print("Scaler fitted only on training data.")
    print("Scaled X min/max:", float(train_df[model_features].min().min()), float(train_df[model_features].max().max()))
    print("Scaled y min/max:", float(train_df[TARGET].min()), float(train_df[TARGET].max()))

    return {
        "train_df": train_df,
        "val_df": val_df,
        "test_df": test_df,
        "model_features": model_features,
        "input_size": len(model_features),
        "future_target_reference": future_target_reference,
    }


### 28.3 Sequence Creation and DataLoaders


In [ ]:
def create_sequences(data, model_features, target_feature, output_seq_length):
    X_parts, y_parts = [], []
    target_index = model_features.index(target_feature)
    data = data.sort_values([STATION_COLUMN, SEGMENT_COLUMN, "timestamp"])

    for _, segment_data in data.groupby([STATION_COLUMN, SEGMENT_COLUMN], sort=True):
        values = np.ascontiguousarray(segment_data[model_features].to_numpy(dtype=np.float32))
        required_length = INPUT_SEQ_LENGTH + output_seq_length
        created_count = max(0, len(values) - required_length + 1)
        if len(values) <= required_length:
            continue
        input_windows = sliding_window_view(values, window_shape=INPUT_SEQ_LENGTH, axis=0)
        input_windows = input_windows[:created_count].transpose(0, 2, 1)
        target_values = values[INPUT_SEQ_LENGTH:, target_index]
        target_windows = sliding_window_view(target_values, window_shape=output_seq_length)[:created_count]
        X_parts.append(np.ascontiguousarray(input_windows, dtype=np.float32))
        y_parts.append(np.ascontiguousarray(target_windows, dtype=np.float32))

    if not X_parts:
        raise ValueError("No sequences were created. Reduce input/output length or inspect split sizes.")
    return np.concatenate(X_parts, axis=0), np.concatenate(y_parts, axis=0)


def create_loader(X, y, shuffle=False):
    X_tensor = torch.from_numpy(X).float()
    y_tensor = torch.from_numpy(y).float()
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=BATCH_SIZE, shuffle=shuffle)


def prepare_data(prepared_data, target_feature, output_seq_length):
    model_features = prepared_data["model_features"]
    X_train, y_train = create_sequences(prepared_data["train_df"], model_features, target_feature, output_seq_length)
    X_val, y_val = create_sequences(prepared_data["val_df"], model_features, target_feature, output_seq_length)
    X_test, y_test = create_sequences(prepared_data["test_df"], model_features, target_feature, output_seq_length)
    return (
        create_loader(X_train, y_train, shuffle=True),
        create_loader(X_val, y_val),
        create_loader(X_test, y_test),
        prepared_data["input_size"],
    )


### 28.4 CNN-LSTM Model


In [ ]:
class CNNLSTMModel(nn.Module):
    def __init__(self, input_size, output_seq_length):
        super().__init__()
        self.conv1 = nn.Conv1d(input_size, CONV_CHANNELS, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(CONV_CHANNELS, CONV_CHANNELS, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(DROPOUT_RATE)
        self.lstm = nn.LSTM(
            CONV_CHANNELS,
            HIDDEN_SIZE,
            NUM_LAYERS,
            batch_first=True,
            dropout=DROPOUT_RATE if NUM_LAYERS > 1 else 0,
        )
        self.fc = nn.Linear(HIDDEN_SIZE, output_seq_length)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.dropout(self.relu(self.conv1(x)))
        x = self.dropout(self.relu(self.conv2(x)))
        x = x.transpose(1, 2)
        output, _ = self.lstm(x)
        output = self.dropout(output[:, -1, :])
        return self.fc(output)


def get_training_device():
    if DEVICE_PREFERENCE == "mps" and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if DEVICE_PREFERENCE == "cuda" and torch.cuda.is_available():
        return torch.device("cuda")
    if DEVICE_PREFERENCE == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")
    return torch.device("cpu")


### 28.5 Training and Evaluation


In [ ]:
def evaluate_loss(model, loader, criterion):
    model.eval()
    device = next(model.parameters()).device
    total_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            predictions = model(X_batch)
            total_loss += criterion(predictions, y_batch).item()
    return total_loss / len(loader)


def train_model(model, train_loader, val_loader, progress_label=""):
    device = get_training_device()
    model.to(device)
    print("Training device:", device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1, min_lr=5e-6)
    train_losses, val_losses = [], []
    best_val_loss = float("inf")
    best_model_state = None

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        val_loss = evaluate_loss(model, val_loader, criterion)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        prefix = f"{progress_label} | " if progress_label else ""
        print(f"{prefix}Epoch {epoch + 1}/{EPOCHS} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())

    if RESTORE_BEST_MODEL and best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model, train_losses, val_losses


def calculate_horizon_metrics(actuals, predictions):
    rows = []
    for step_index in range(actuals.shape[1]):
        step_actuals = actuals[:, step_index]
        step_predictions = predictions[:, step_index]
        mse = mean_squared_error(step_actuals, step_predictions)
        mae = mean_absolute_error(step_actuals, step_predictions)
        rmse = np.sqrt(mse)
        r2 = r2_score(step_actuals, step_predictions)
        rows.append({"forecast_step": step_index + 1, "mse": mse, "mae": mae, "rmse": rmse, "r2": r2})
    return pd.DataFrame(rows)


def evaluate_model(model, test_loader):
    model.eval()
    device = next(model.parameters()).device
    predictions, actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(y_batch.numpy())
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    mse = mean_squared_error(actuals.flatten(), predictions.flatten())
    mae = mean_absolute_error(actuals.flatten(), predictions.flatten())
    rmse = np.sqrt(mse)
    r2 = r2_score(actuals.flatten(), predictions.flatten())
    return predictions, actuals, mse, mae, rmse, r2, calculate_horizon_metrics(actuals, predictions)


### 28.6 Plot Saving


In [ ]:
def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))


def save_loss_plot(train_losses, val_losses, model_name, target_feature, output_seq_length):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} Train vs Validation Loss - {target_feature} | OL {output_seq_length}")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    path = OUTPUT_DIR / f"{safe_name(model_name)}_{safe_name(target_feature)}_OL{output_seq_length}_train_validation_loss.png"
    plt.savefig(path, dpi=300)
    if SHOW_PLOTS_IN_NOTEBOOK:
        plt.show()
    else:
        plt.close()
    return path


def save_prediction_plot(actuals, predictions, model_name, target_feature, output_seq_length):
    if target_feature == TARGET and PLOT_ALL_TIMESTAMPS_FOR_CO2:
        max_points = len(actuals)
    else:
        max_points = min(PLOT_MAX_TIMESTAMPS, len(actuals))
    step_index = output_seq_length - 1
    plt.figure(figsize=(10, 4))
    plt.plot(actuals[:max_points, step_index], label="Actual")
    plt.plot(predictions[:max_points, step_index], label="Predicted")
    plt.xlabel("Timestamp index")
    plt.ylabel(f"Scaled {target_feature}")
    plt.title(f"{model_name} Actual vs Predicted - {target_feature} | OL {output_seq_length}")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    path = OUTPUT_DIR / f"{safe_name(model_name)}_{safe_name(target_feature)}_OL{output_seq_length}_actual_vs_predicted.png"
    plt.savefig(path, dpi=300)
    if SHOW_PLOTS_IN_NOTEBOOK:
        plt.show()
    else:
        plt.close()
    return path


### 28.7 Run Selected Feature Batch


In [ ]:
def run_one_experiment(prepared_data, model_name, model_class, target_feature, output_seq_length, run_index, total_runs):
    print("\n" + "=" * 72)
    progress_label = f"Run {run_index}/{total_runs} | target={target_feature} | horizon={output_seq_length}"
    print(f"{model_name} metrics run | {progress_label}")
    print("=" * 72)

    train_loader, val_loader, test_loader, input_size = prepare_data(prepared_data, target_feature, output_seq_length)
    model = model_class(input_size, output_seq_length)
    model, train_losses, val_losses = train_model(model, train_loader, val_loader, progress_label)
    predictions, actuals, mse, mae, rmse, r2, horizon_metrics = evaluate_model(model, test_loader)

    loss_plot = save_loss_plot(train_losses, val_losses, model_name, target_feature, output_seq_length)
    prediction_plot = save_prediction_plot(actuals, predictions, model_name, target_feature, output_seq_length)

    horizon_metrics["model"] = model_name
    horizon_metrics["target_feature"] = target_feature
    horizon_metrics["output_length"] = output_seq_length

    return {
        "Model": model_name,
        "Target_Feature": target_feature,
        "Output_Length": output_seq_length,
        "MSE": mse,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Input_Length": INPUT_SEQ_LENGTH,
        "Batch_Size": BATCH_SIZE,
        "Learning_Rate": LEARNING_RATE,
        "Input_Features": input_size,
        "Train_Batches": len(train_loader),
        "Val_Batches": len(val_loader),
        "Test_Batches": len(test_loader),
        "Loss_Plot": str(loss_plot),
        "Prediction_Plot": str(prediction_plot),
    }, horizon_metrics


raw_df = load_prepare_data(CSV_PATH)
print("Number of LSTM layers:", NUM_LAYERS)
print("Maximum feature fill steps:", MAX_FILL_STEPS)
print("Drop short stations:", DROP_SHORT_STATIONS)
print("Clip outliers:", CLIP_OUTLIERS)
print("Outlier clip factor:", OUTLIER_CLIP_FACTOR)
print("Restore best validation checkpoint:", RESTORE_BEST_MODEL)
print("Gap-aware sequence generation:", USE_GAP_AWARE_SEGMENTS)
print("Convolution channels:", CONV_CHANNELS)
print("Use scattering:", USE_SCATTERING)
print("Use attention:", USE_ATTENTION)

prepared_data = prepare_base_data(raw_df)
missing_targets = [target for target in RUN_TARGET_FEATURES if target not in raw_df.columns]
if missing_targets:
    raise ValueError(f"Selected target features missing from CSV: {missing_targets}")

all_results_summary = []
all_horizon_metrics = []
total_runs = len(RUN_TARGET_FEATURES) * len(OUTPUT_LENGTHS)
print("\nSelected target features:", RUN_TARGET_FEATURES)
print(f"Total runs: {total_runs} ({len(RUN_TARGET_FEATURES)} features x {len(OUTPUT_LENGTHS)} horizons)")

run_index = 0
for target_feature in RUN_TARGET_FEATURES:
    for output_seq_length in OUTPUT_LENGTHS:
        run_index += 1
        result_row, horizon_metrics = run_one_experiment(
            prepared_data,
            'CNN-LSTM',
            CNNLSTMModel,
            target_feature,
            output_seq_length,
            run_index,
            total_runs,
        )
        all_results_summary.append(result_row)
        all_horizon_metrics.append(horizon_metrics)

summary_df = pd.DataFrame(all_results_summary)
metrics_path = OUTPUT_DIR / f"{safe_name('CNN-LSTM')}_feature_metrics_by_horizon.csv"
summary_df.to_csv(metrics_path, index=False)
print("\nSaved metrics summary:", metrics_path)

horizon_metrics_df = pd.concat(all_horizon_metrics, ignore_index=True) if all_horizon_metrics else pd.DataFrame()
horizon_metrics_path = OUTPUT_DIR / f"{safe_name('CNN-LSTM')}_per_forecast_step_metrics.csv"
horizon_metrics_df.to_csv(horizon_metrics_path, index=False)
print("Saved per-step metrics:", horizon_metrics_path)
summary_df
